In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# ============================================================
# Fine-Tuning Helsinki-NLP/opus-mt-en-hi (MarianMT)
# Dataset: Dataset_English_Hindi.csv (English, Hindi columns)
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    MarianTokenizer,
    MarianMTModel,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

C:\Users\Dex\anaconda3\envs\pytorch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────
MODEL_NAME       = "Helsinki-NLP/opus-mt-en-hi"
DATA_PATH        = "Dataset_English_Hindi.csv"
OUTPUT_DIR       = "./opus-mt-en-hi-finetuned"
MAX_INPUT_LEN    = 128
MAX_TARGET_LEN   = 128
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE  = 32
NUM_EPOCHS       = 8
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
TEST_SIZE        = 0.1       # 10% validation split
SEED             = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
# ─────────────────────────────────────────────
# 2. LOAD & CLEAN DATASET
# ─────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()                 # strip whitespace from column names
df = df[["English", "Hindi"]].dropna()              # keep only relevant columns
df = df[df["English"].str.strip() != ""]            # drop empty English rows
df = df[df["Hindi"].str.strip() != ""]              # drop empty Hindi rows
df = df.drop_duplicates()                           # remove duplicate pairs
df = df.reset_index(drop=True)

print(f"Total sentence pairs after cleaning: {len(df):,}")
print(df.head(5))

Total sentence pairs after cleaning: 127,375
  English    Hindi
0   Help!    बचाओ!
1   Jump.    उछलो.
2   Jump.    कूदो.
3   Jump.   छलांग.
4  Hello!  नमस्ते।


In [5]:
# ─────────────────────────────────────────────
# 3. TRAIN / VALIDATION SPLIT
# ─────────────────────────────────────────────
train_df, val_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, shuffle=True
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,}")

# Convert to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

Train size: 114,637 | Val size: 12,738


In [6]:
# ─────────────────────────────────────────────
# 4. TOKENIZER & MODEL
# ─────────────────────────────────────────────
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model     = MarianMTModel.from_pretrained(MODEL_NAME).to(device)

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")


Model loaded: Helsinki-NLP/opus-mt-en-hi
Vocabulary size: 61,950


In [7]:
# ─────────────────────────────────────────────
# 5. TOKENIZATION FUNCTION
# ─────────────────────────────────────────────
def preprocess(batch):
    """Tokenize source (English) and target (Hindi) sentences."""
    model_inputs = tokenizer(
        batch["English"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,          # DataCollator handles padding dynamically
    )
    # Tokenize targets using text_target argument (preferred in newer transformers)
    labels = tokenizer(
        text_target=batch["Hindi"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization (batched for speed)
tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=["English", "Hindi"],
    desc="Tokenizing train set",
)
tokenized_val = val_dataset.map(
    preprocess,
    batched=True,
    remove_columns=["English", "Hindi"],
    desc="Tokenizing val set",
)

print("\nTokenized train sample keys:", tokenized_train.column_names)

Tokenizing val set: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12738/12738 [00:01<00:00, 6981.48 examples/s]


Tokenized train sample keys: ['input_ids', 'attention_mask', 'labels']


In [8]:
# ─────────────────────────────────────────────
# 6. DATA COLLATOR  (dynamic padding)
# ─────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,       # speeds up training on Tensor Cores
    label_pad_token_id=-100,    # ignore padding tokens in loss
)

In [9]:
# ─────────────────────────────────────────────
# 7. METRICS  (BLEU via sacrebleu)
# ─────────────────────────────────────────────
sacrebleu = evaluate.load("sacrebleu")

def postprocess(preds, labels):
    """Strip padding and convert token IDs back to strings."""
    preds  = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]   # sacrebleu expects list of references
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Decode predictions — replace -100 labels with pad_token_id first
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess(decoded_preds, decoded_labels)

    result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    avg_len = np.mean([np.count_nonzero(p != tokenizer.pad_token_id) for p in preds])

    return {
        "bleu":    round(result["score"], 4),
        "gen_len": round(avg_len, 4),
    }

In [10]:
# ─────────────────────────────────────────────
# 8. TRAINING ARGUMENTS
# ─────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,

    per_device_train_batch_size = TRAIN_BATCH_SIZE,
    per_device_eval_batch_size  = EVAL_BATCH_SIZE,

    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = "cosine",         # smooth LR decay
    warmup_ratio                = 0.05,             # 5% warmup steps

    # Evaluation & checkpointing
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,                # keep only 2 best checkpoints
    load_best_model_at_end      = True,
    metric_for_best_model       = "bleu",
    greater_is_better           = True,

    # Generation settings for eval
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET_LEN,

    # Logging
    logging_dir                 = f"{OUTPUT_DIR}/logs",
    logging_strategy            = "epoch",
    report_to                   = "none",           # change to "tensorboard" if needed

    # Efficiency
    fp16                        = torch.cuda.is_available(),  # AMP on GPU
    gradient_accumulation_steps = 2,               # effective batch = 32
    dataloader_num_workers      = 2,

    seed                        = SEED,
)

In [11]:
# ─────────────────────────────────────────────
# 9. TRAINER
# ─────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_train,
    eval_dataset    = tokenized_val,
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

In [12]:
# ─────────────────────────────────────────────
# 10. TRAIN
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("Starting fine-tuning...")
print("="*50)

trainer.train()


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Bleu,Gen Len
0,3.955900,3.406237,10.110200,26.834200
2,3.116500,3.009869,12.153600,23.910900
4,2.836900,2.879817,12.999000,23.212700
6,2.728600,2.843662,13.280900,23.057800
7,2.712100,2.842858,13.289500,23.087100


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61949]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61949]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strate

TrainOutput(global_step=14328, training_loss=3.0543984094190573, metrics={'train_runtime': 13754.1555, 'train_samples_per_second': 66.678, 'train_steps_per_second': 1.042, 'total_flos': 1.990183386297139e+16, 'train_loss': 3.0543984094190573, 'epoch': 7.9977672341613175})

In [13]:
# ─────────────────────────────────────────────
# 11. SAVE BEST MODEL & TOKENIZER
# ─────────────────────────────────────────────
best_model_path = f"{OUTPUT_DIR}/best_model"
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"\nBest model saved to: {best_model_path}")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61949]], 'forced_eos_token_id': 0}



Best model saved to: ./opus-mt-en-hi-finetuned/best_model


In [14]:
# ─────────────────────────────────────────────
# 12. EVALUATE ON VALIDATION SET
# ─────────────────────────────────────────────
eval_results = trainer.evaluate()
print("\n── Validation Metrics ──")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


── Validation Metrics ──
  eval_loss: 2.8429
  eval_bleu: 13.2895
  eval_gen_len: 23.0871
  eval_runtime: 1179.7475
  eval_samples_per_second: 10.7970
  eval_steps_per_second: 0.3380
  epoch: 7.9978


In [15]:
# ─────────────────────────────────────────────
# 13. INFERENCE HELPER
# ─────────────────────────────────────────────
def translate(text: str | list[str], model_path: str = best_model_path) -> list[str]:
    """
    Translate one or multiple English sentences to Hindi.

    Args:
        text       : A single string or a list of English strings.
        model_path : Path to the saved fine-tuned model.

    Returns:
        List of Hindi translated strings.
    """
    if isinstance(text, str):
        text = [text]

    inf_tokenizer = MarianTokenizer.from_pretrained(model_path)
    inf_model     = MarianMTModel.from_pretrained(model_path).to(device)
    inf_model.eval()

    inputs = inf_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN,
    ).to(device)

    with torch.no_grad():
        translated_ids = inf_model.generate(
            **inputs,
            num_beams          = 4,
            max_length         = MAX_TARGET_LEN,
            early_stopping     = True,
            no_repeat_ngram_size = 3,
        )

    translations = inf_tokenizer.batch_decode(translated_ids, skip_special_tokens=True)
    return translations

In [16]:
# ─────────────────────────────────────────────
# 14. QUICK TEST
# ─────────────────────────────────────────────
test_sentences = [
    "Hello, how are you?",
    "I love learning new languages.",
    "The weather is very nice today.",
    "Can you help me find the nearest hospital?",
    "She is reading a book in the library.",
]

print("\n── Translation Samples ──")
results = translate(test_sentences)
for en, hi in zip(test_sentences, results):
    print(f"  EN: {en}")
    print(f"  HI: {hi}\n")


── Translation Samples ──
  EN: Hello, how are you?
  HI: हैलो, तुम कैसे हो?

  EN: I love learning new languages.
  HI: मुझे नई भाषा सीखने में बहुत मज़ा आता है।

  EN: The weather is very nice today.
  HI: मौसम आज बहुत अच्छा है।

  EN: Can you help me find the nearest hospital?
  HI: क्या आप मेरी मदद कर सकते हैं मेरे अस्पताल में?

  EN: She is reading a book in the library.
  HI: वह पुस्तकालय में एक किताब पढ़ रही है।



In [17]:
test_sentences = [
    "How are you doing today",
    "I want to eat food",
    "The weather is very cold outside",
    "She is going to the market",
    "My name is Rahul",
    "I love my country",
    "He is a very good student",
    "Please open the door",
    "I am feeling very tired today",
    "The dog is running in the park",
    "Can you help me please",
    "I want to drink water",
    "She loves to read books",
    "The sun rises in the east",
    "I go to school every day",
    "He does not like cold weather",
    "We are going to the temple",
    "The train is very late today",
    "I want to sleep now",
    "My mother cooks very delicious food"
]

print("\n── Translation Samples ──")
results = translate(test_sentences)
for en, hi in zip(test_sentences, results):
    print(f"  EN: {en}")
    print(f"  HI: {hi}\n")


── Translation Samples ──
  EN: How are you doing today
  HI: आज आप कैसे कर रहे हैं

  EN: I want to eat food
  HI: मैं खाना चाहता हूँ

  EN: The weather is very cold outside
  HI: मौसम बहुत ठंडा है

  EN: She is going to the market
  HI: वह बाज़ार जा रही है

  EN: My name is Rahul
  HI: मेरा नाम राहुल है।

  EN: I love my country
  HI: मैं अपने देश से प्यार करता हूँ

  EN: He is a very good student
  HI: वह एक बहुत अच्छा विद्यार्थी है

  EN: Please open the door
  HI: कृपया दरवाजा खोल दें

  EN: I am feeling very tired today
  HI: मैं आज बहुत थका हुआ महसूस कर रहा हूँ

  EN: The dog is running in the park
  HI: कुत्ते पार्क में दौड़ रहा है

  EN: Can you help me please
  HI: क्या आप मेरी मदद कर सकते हैं

  EN: I want to drink water
  HI: मैं पानी पीना चाहता हूँ

  EN: She loves to read books
  HI: उसे किताबें पढ़ना अच्छा लगता है

  EN: The sun rises in the east
  HI: पूर्व में सूर्य उदय होता है

  EN: I go to school every day
  HI: मैं रोज़ स्कूल जाता हूँ

  EN: He does not like cold 